In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags
from src.features.batted_ball import add_quality_flags, batted_ball_events

df = load_all_snapshots()
flagged = add_discipline_flags(df)
print(flagged.shape)

(710632, 123)


In [2]:
def split_half_correlation(data, batter_col, num_col, den_col, sample_sizes, seed=42):
    """For each sample size n: give each batter two random disjoint samples
    of n pitches, compute the rate in each half, and correlate across batters.

    A metric is 'stable' at n when the two halves agree — meaning the
    number reflects the player, not the draw.
    """
    rng = np.random.default_rng(seed)
    results = []

    for n in sample_sizes:
        first, second = [], []
        for _, g in data.groupby(batter_col):
            elig = g[g[den_col]]
            if len(elig) < 2 * n:
                continue
            idx = rng.permutation(len(elig))
            a = elig.iloc[idx[:n]]
            b = elig.iloc[idx[n:2 * n]]
            first.append(a[num_col].mean())
            second.append(b[num_col].mean())

        if len(first) >= 20:
            r = np.corrcoef(first, second)[0, 1]
            results.append({"n": n, "batters": len(first), "r": round(r, 3)})

    return pd.DataFrame(results)


flagged["_all"] = True
sizes = [25, 50, 100, 200, 300, 500]

print("=== Chase% (denominator: out-of-zone pitches) ===")
oz = flagged.copy()
oz["_oz"] = ~oz["in_zone"]
print(split_half_correlation(oz, "batter", "is_swing", "_oz", sizes).to_string(index=False))

=== Chase% (denominator: out-of-zone pitches) ===
  n  batters     r
 25      581 0.358
 50      523 0.491
100      450 0.610
200      361 0.754
300      278 0.798
500      128 0.850


In [3]:
iz = flagged.copy()
iz["_iz"] = iz["in_zone"]

zs = flagged.copy()
zs["_zone_swing"] = zs["is_swing"] & zs["in_zone"]

bbe = add_quality_flags(batted_ball_events(df))
bbe["_all"] = True

print("=== Zone Swing% ===")
print(split_half_correlation(iz, "batter", "is_swing", "_iz", sizes).to_string(index=False))
print()
print("=== Zone Contact% (denominator: zone swings) ===")
zs["_contact"] = ~zs["is_whiff"]
print(split_half_correlation(zs, "batter", "_contact", "_zone_swing", sizes).to_string(index=False))
print()
print("=== HardHit% (denominator: BBE) ===")
print(split_half_correlation(bbe, "batter", "is_hard_hit", "_all", [25, 50, 100, 150]).to_string(index=False))
print()
print("=== Barrel% (denominator: BBE) ===")
print(split_half_correlation(bbe, "batter", "is_barrel", "_all", [25, 50, 100, 150]).to_string(index=False))

=== Zone Swing% ===
  n  batters     r
 25      583 0.214
 50      524 0.381
100      452 0.573
200      361 0.725
300      269 0.824
500      115 0.872

=== Zone Contact% (denominator: zone swings) ===
  n  batters     r
 25      548 0.257
 50      489 0.450
100      403 0.647
200      273 0.782
300      158 0.842

=== HardHit% (denominator: BBE) ===
  n  batters     r
 25      485 0.334
 50      405 0.525
100      284 0.715
150      167 0.801

=== Barrel% (denominator: BBE) ===
  n  batters     r
 25      485 0.232
 50      405 0.468
100      284 0.650
150      167 0.741


In [4]:
from src.features.plate_discipline import discipline_profile
from src.features.batted_ball import quality_profile
from src.data.player_ids import load_player_ids, display_name

def batter_profile(g):
    p = discipline_profile(g)
    p.update(quality_profile(g))
    return p

MIN_PITCHES = 500
rows = []
for bid, g in df.groupby("batter"):
    if len(g) < MIN_PITCHES:
        continue
    p = batter_profile(g)
    p["batter"] = bid
    rows.append(p)

prof = pd.DataFrame(rows).set_index("batter")
ids = load_player_ids(prof.index.tolist())
prof = prof.join(display_name(ids))

print(prof.shape)
print(prof.columns.tolist())

(425, 20)
['pitches', 'zone_pct', 'swing_pct', 'chase_pct', 'zone_swing_pct', 'contact_pct', 'zone_contact_pct', 'whiff_pct', 'n_out_of_zone', 'n_in_zone', 'n_swings', 'n_zone_swings', 'bbe', 'barrel_pct', 'hard_hit_pct', 'sweet_spot_pct', 'avg_exit_velocity', 'max_exit_velocity', 'avg_launch_angle', 'name']


In [5]:
cols = ["name", "chase_pct", "zone_contact_pct", "barrel_pct", "hard_hit_pct", "bbe"]
print("=== highest barrel rate (min 100 BBE) ===")
print(prof[prof["bbe"] >= 100].nlargest(10, "barrel_pct")[cols].round(3).to_string())
print()
print("=== correlation matrix ===")
metrics = ["chase_pct", "zone_swing_pct", "zone_contact_pct",
           "barrel_pct", "hard_hit_pct", "sweet_spot_pct"]
print(prof[metrics].corr().round(2).to_string())

=== highest barrel rate (min 100 BBE) ===
                      name  chase_pct  zone_contact_pct  barrel_pct  hard_hit_pct  bbe
batter                                                                                
592450        Judge, Aaron      0.179             0.796       0.270         0.614  389
660271      Ohtani, Shohei      0.259             0.803       0.218         0.605  468
519317  Stanton, Giancarlo      0.309             0.779       0.209         0.557  273
665742          Soto, Juan      0.178             0.855       0.198         0.570  460
681481    Carpenter, Kerry      0.326             0.832       0.178         0.466  191
670242       Wallner, Matt      0.259             0.710       0.175         0.532  126
641933      O'Neill, Tyler      0.257             0.769       0.173         0.488  254
669911     Toglia, Michael      0.270             0.796       0.173         0.502  255
669357       Gorman, Nolan      0.302             0.735       0.167         0.386  215
6